# Meta-analysis: pooling results across experiments

This notebook pools results from all three A/B test experiments to
estimate overall treatment effects, assess heterogeneity, generate
a forest plot, and build a decision framework that distinguishes
statistical significance from practical significance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
sys.path.insert(0, '..')

from src.experiment import frequentist_test, bayesian_ab, summary_report
from src.visualizations import plot_forest

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Collect results from all experiments

In [ ]:
df = pd.read_csv('../data/ab_test_results.csv')

all_reports = []
for exp_id in df['experiment_id'].unique():
    exp_data = df[df['experiment_id'] == exp_id]
    report = summary_report(exp_data, experiment_id=exp_id)
    all_reports.append(report)

# Build summary table
summary = []
for r in all_reports:
    fc = r['frequentist_conversion']
    summary.append({
        'experiment_id': r['experiment_id'],
        'n_control': r['n_control'],
        'n_treatment': r['n_treatment'],
        'control_rate': r['control_conversion_rate'],
        'treatment_rate': r['treatment_conversion_rate'],
        'effect': fc['effect'],
        'ci_lower': fc['ci_lower'],
        'ci_upper': fc['ci_upper'],
        'p_value': fc['p_value'],
        'significant': fc['significant'],
        'effect_size_h': fc['effect_size'],
    })

summary_df = pd.DataFrame(summary)
print('Experiment results summary:')
summary_df

## 2. Forest plot

A forest plot displays the effect size and confidence interval for
each experiment on a common scale. It makes it easy to see which
experiments found significant effects (CI excludes zero) and how
heterogeneous the effects are.

In [ ]:
# Use the visualization module
plot_forest(all_reports)
plt.show()

In [ ]:
# Manual forest plot with more detail
fig, ax = plt.subplots(figsize=(10, 4))

y_positions = range(len(summary_df))
labels = summary_df['experiment_id'].tolist()
effects = summary_df['effect'].values
ci_lo = summary_df['ci_lower'].values
ci_hi = summary_df['ci_upper'].values

for i in range(len(summary_df)):
    color = '#1565C0' if summary_df.iloc[i]['significant'] else '#90CAF9'
    ax.plot([ci_lo[i], ci_hi[i]], [i, i], color=color, linewidth=2.5)
    ax.plot(effects[i], i, 'o', color=color, markersize=10, zorder=5)
    # Annotate
    ax.text(ci_hi[i] + 0.002, i,
            f'{effects[i]:+.4f} [{ci_lo[i]:.4f}, {ci_hi[i]:.4f}]',
            va='center', fontsize=9)

ax.axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_yticks(y_positions)
ax.set_yticklabels(labels)
ax.set_xlabel('Effect size (treatment - control)')
ax.set_title('Forest plot: effect sizes with 95% confidence intervals')
ax.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1565C0', label='Significant (p < 0.05)'),
    Patch(facecolor='#90CAF9', label='Not significant'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 3. Pooled effect estimation (fixed-effects meta-analysis)

We estimate the overall treatment effect using an inverse-variance
weighted average across experiments. This gives more weight to
experiments with larger samples and smaller variances.

In [ ]:
# Compute standard errors for each experiment
se_list = []
for _, row in summary_df.iterrows():
    p_c = row['control_rate']
    p_t = row['treatment_rate']
    n_c = row['n_control']
    n_t = row['n_treatment']
    se = np.sqrt(p_c * (1 - p_c) / n_c + p_t * (1 - p_t) / n_t)
    se_list.append(se)

summary_df['se'] = se_list
summary_df['weight'] = 1 / summary_df['se']**2

# Fixed-effects pooled estimate
pooled_effect = (summary_df['effect'] * summary_df['weight']).sum() / summary_df['weight'].sum()
pooled_se = np.sqrt(1 / summary_df['weight'].sum())
pooled_ci = (pooled_effect - 1.96 * pooled_se, pooled_effect + 1.96 * pooled_se)
pooled_z = pooled_effect / pooled_se
pooled_p = 2 * (1 - stats.norm.cdf(abs(pooled_z)))

print('Fixed-effects meta-analysis:')
print(f'  Pooled effect:  {pooled_effect:+.5f}')
print(f'  95% CI:         [{pooled_ci[0]:.5f}, {pooled_ci[1]:.5f}]')
print(f'  Z-statistic:    {pooled_z:.3f}')
print(f'  p-value:        {pooled_p:.6f}')
print(f'  Significant:    {pooled_p < 0.05}')

print(f'\nPer-experiment weights:')
for _, row in summary_df.iterrows():
    pct = row['weight'] / summary_df['weight'].sum() * 100
    print(f'  {row["experiment_id"]}: {pct:.1f}%')

## 4. Heterogeneity assessment

Cochran's Q test and the I-squared statistic measure whether the
effect sizes across experiments are more variable than expected
from sampling variation alone.

In [ ]:
# Cochran's Q statistic
Q = (summary_df['weight'] * (summary_df['effect'] - pooled_effect)**2).sum()
k = len(summary_df)  # number of studies
df_q = k - 1
q_p_value = 1 - stats.chi2.cdf(Q, df_q)

# I-squared
I_squared = max(0, (Q - df_q) / Q * 100) if Q > 0 else 0

print('Heterogeneity assessment:')
print(f'  Cochran\'s Q:     {Q:.3f}')
print(f'  df:               {df_q}')
print(f'  Q p-value:        {q_p_value:.4f}')
print(f'  I-squared:        {I_squared:.1f}%')
print()

if I_squared < 25:
    heterogeneity = 'low'
elif I_squared < 50:
    heterogeneity = 'moderate'
elif I_squared < 75:
    heterogeneity = 'substantial'
else:
    heterogeneity = 'considerable'

print(f'Interpretation: {heterogeneity} heterogeneity')
print(f'I-squared thresholds: <25% low, 25-50% moderate, 50-75% substantial, >75% considerable')

if I_squared > 50:
    print('\nHigh heterogeneity suggests the experiments are measuring different')
    print('underlying effects. A random-effects model would be more appropriate.')
else:
    print('\nHeterogeneity is within acceptable bounds for a fixed-effects model.')

In [ ]:
# Visualize heterogeneity: effect sizes with pooled estimate
fig, ax = plt.subplots(figsize=(8, 4))

experiment_names = summary_df['experiment_id'].tolist()
x_pos = range(len(experiment_names))

ax.bar(x_pos, summary_df['effect'].values, color='steelblue', alpha=0.7,
       yerr=[summary_df['effect'] - summary_df['ci_lower'],
             summary_df['ci_upper'] - summary_df['effect']],
       capsize=8, label='Per-experiment effect')
ax.axhline(pooled_effect, color='red', linestyle='-', linewidth=2,
           label=f'Pooled effect ({pooled_effect:+.4f})')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(experiment_names)
ax.set_ylabel('Effect size')
ax.set_title(f'Effect sizes vs pooled estimate (I2={I_squared:.0f}%)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Effect size estimation and interpretation

Effect sizes provide a standardized measure of treatment impact
that is independent of sample size. Cohen's h is the appropriate
measure for comparing proportions.

In [ ]:
print('Effect size interpretation (Cohen\'s h for proportions):')
print(f'  Small:  h ~ 0.2')
print(f'  Medium: h ~ 0.5')
print(f'  Large:  h ~ 0.8')
print()

for _, row in summary_df.iterrows():
    h = abs(row['effect_size_h'])
    if h < 0.2:
        size_label = 'negligible'
    elif h < 0.5:
        size_label = 'small'
    elif h < 0.8:
        size_label = 'medium'
    else:
        size_label = 'large'

    print(f'{row["experiment_id"]}:')
    print(f'  Absolute effect: {row["effect"]:+.4f} ({row["effect"]/row["control_rate"]:+.1%} relative)')
    print(f'  Cohen\'s h:       {row["effect_size_h"]:.4f} ({size_label})')
    print(f'  Significant:     {row["significant"]}')
    print()

## 6. Practical significance vs statistical significance

A result can be statistically significant but not practically meaningful,
or vice versa. The decision to ship a change should consider both.

In [ ]:
# Define practical significance thresholds
min_practical_effect = 0.005  # 0.5pp minimum meaningful lift

fig, ax = plt.subplots(figsize=(10, 5))

for i, (_, row) in enumerate(summary_df.iterrows()):
    color = '#1565C0' if row['significant'] else '#90CAF9'
    marker = 's' if abs(row['effect']) >= min_practical_effect else 'o'

    ax.plot([row['ci_lower'], row['ci_upper']], [i, i],
            color=color, linewidth=3)
    ax.plot(row['effect'], i, marker, color=color, markersize=12, zorder=5)

    # Quadrant label
    stat_sig = row['significant']
    prac_sig = abs(row['effect']) >= min_practical_effect
    if stat_sig and prac_sig:
        label = 'SHIP'
    elif stat_sig and not prac_sig:
        label = 'Significant but small'
    elif not stat_sig and prac_sig:
        label = 'Underpowered'
    else:
        label = 'No action'
    ax.text(max(row['ci_upper'] + 0.003, 0.04), i, label,
            va='center', fontsize=9, fontstyle='italic')

ax.axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(min_practical_effect, color='green', linestyle=':',
           linewidth=1.5, alpha=0.7, label=f'Min practical effect ({min_practical_effect:.1%})')
ax.axvline(-min_practical_effect, color='green', linestyle=':',
           linewidth=1.5, alpha=0.7)

ax.set_yticks(range(len(summary_df)))
ax.set_yticklabels(summary_df['experiment_id'])
ax.set_xlabel('Effect size (treatment - control)')
ax.set_title('Statistical vs practical significance')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Business decision framework

We formalize the decision process into a structured framework
that combines statistical evidence with business context.

In [ ]:
avg_order_value = 45.0
monthly_traffic = 100000

decisions = []
for _, row in summary_df.iterrows():
    effect = row['effect']
    stat_sig = row['significant']
    prac_sig = abs(effect) >= min_practical_effect

    # Revenue impact estimate
    monthly_rev_impact = monthly_traffic * effect * avg_order_value
    annual_rev_impact = monthly_rev_impact * 12

    # Decision logic
    if stat_sig and prac_sig and effect > 0:
        decision = 'Ship the treatment'
        confidence = 'High'
    elif stat_sig and not prac_sig:
        decision = 'Statistically significant but effect too small to justify costs'
        confidence = 'Medium'
    elif not stat_sig and prac_sig:
        decision = 'Re-run with larger sample (experiment was underpowered)'
        confidence = 'Low'
    else:
        decision = 'No action -- no evidence of meaningful effect'
        confidence = 'High (to not ship)'

    decisions.append({
        'Experiment': row['experiment_id'],
        'Effect': f'{effect:+.4f}',
        'Stat. sig.': stat_sig,
        'Prac. sig.': prac_sig,
        'Annual rev. impact': f'${annual_rev_impact:+,.0f}',
        'Decision': decision,
        'Confidence': confidence,
    })

decision_df = pd.DataFrame(decisions).set_index('Experiment')
print('Business decision framework:')
decision_df

In [ ]:
# Revenue impact waterfall
fig, ax = plt.subplots(figsize=(8, 4))

exp_names = summary_df['experiment_id'].tolist()
rev_impacts = [
    monthly_traffic * row['effect'] * avg_order_value * 12
    for _, row in summary_df.iterrows()
]

colors = ['#4CAF50' if r > 0 else '#FF5722' for r in rev_impacts]
ax.bar(exp_names, rev_impacts, color=colors)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Estimated annual revenue impact ($)')
ax.set_title('Projected annual revenue impact per experiment')
ax.grid(True, alpha=0.3, axis='y')

for i, (name, rev) in enumerate(zip(exp_names, rev_impacts)):
    ax.text(i, rev, f'${rev:+,.0f}', ha='center',
            va='bottom' if rev > 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

total_impact = sum(r for r in rev_impacts if summary_df.iloc[i]['significant'])
print(f'Total projected annual impact from significant experiments: ${total_impact:+,.0f}')

## 8. Decision matrix

The 2x2 matrix of statistical and practical significance creates
four quadrants that map to clear actions.

In [ ]:
print('Decision matrix:')
print()
print('                     Practically significant    Not practically significant')
print('                     -----------------------    ---------------------------')
print('  Stat. significant  | SHIP the change       | Monitor -- real but tiny   |')
print('                     | High confidence        | May not justify eng. cost  |')
print('                     -----------------------    ---------------------------')
print('  Not stat. sig.     | Re-run with more data | NO ACTION                  |')
print('                     | Likely underpowered    | No evidence of any effect  |')
print('                     -----------------------    ---------------------------')
print()
print('Key principles:')
print('  1. Never ship based solely on statistical significance')
print('  2. Always estimate the business impact in dollar terms')
print('  3. Consider the cost of implementation when deciding')
print('  4. Underpowered experiments deserve a re-run, not a dismissal')
print('  5. Pre-register the decision criteria before the experiment starts')

## Summary

The meta-analysis across three experiments reveals:

- **Pooled effect**: a small positive overall treatment effect, driven
  primarily by exp_001 and exp_003
- **Heterogeneity**: the experiments measure different types of interventions,
  so we should interpret the pooled estimate cautiously
- **Forest plot**: provides a clear visual summary of where effects are
  significant and where confidence intervals overlap zero
- **Practical significance**: exp_001 and exp_003 have effects large enough
  to justify shipping; exp_002 does not
- **Business framework**: combining statistical evidence with revenue impact
  estimates produces actionable decisions for each experiment